# Análisis de similitud de imágenes con CLIP y FAISS

## Objetivo
Este notebook implementa un flujo de análisis de imágenes basado en embeddings
generados con el modelo CLIP y búsqueda de similitud mediante FAISS.
El objetivo es permitir la recuperación de imágenes visualmente similares
a partir de una imagen o una consulta textual, facilitando la exploración
semántica de archivos visuales y colecciones artísticas.

## Contexto del dataset
Para este ejemplo se trabajó con un corpus de aproximadamente **10.700 imágenes**
provenientes de colecciones abiertas de instituciones culturales, entre ellas:

- **The Art Institute of Chicago**
- **Museo del Banco de la República (Colombia)**
- **Museo Thyssen-Bornemisza**

Las imágenes y sus metadatos fueron unificados en un único dataset estructurado,
permitiendo su análisis conjunto independientemente de la institución de origen.

## Diccionario de datos
El dataset utilizado se organiza como un DataFrame donde cada fila representa
una obra y sus metadatos asociados. Las principales columnas son:

- **id**: Identificador único de la obra (clave de enlace con FAISS).
- **source**: Institución o fuente de procedencia de la obra.
- **source_url**: URL original del registro en la institución fuente.
- **source_url_sha1**: Hash SHA-1 de la URL fuente, usado para deduplicación.
- **inventory_no**: Número de inventario institucional.
- **collection**: Colección curatorial a la que pertenece la obra.
- **room**: Sala o sección de exhibición (cuando aplica).
- **location**: Ubicación física o curatorial de la obra.
- **title**: Título original de la obra.
- **artist**: Autor o artista asociado.
- **date_text**: Fecha de la obra en formato textual.
- **year_start / year_end**: Rango temporal estimado de producción.
- **technique**: Técnica artística empleada.
- **medium**: Materiales o soporte de la obra.
- **dimensions**: Dimensiones físicas.
- **size_text**: Descripción textual del tamaño.
- **description**: Descripción curatorial o interpretativa.
- **info_extra**: Información adicional relevante.
- **bibliography**: Referencias bibliográficas asociadas.
- **exhibition_hist**: Historial de exposiciones.
- **image_url**: URL de la imagen estándar.
- **image_full_url**: URL de la imagen en alta resolución.
- **raw_json**: Registro original en formato JSON.
- **scraped_at**: Fecha de recolección del dato.
- **updated_at**: Fecha de última actualización.
- **title_norm**: Título normalizado para búsquedas.
- **artist_norm**: Nombre del artista normalizado.

## Tecnologías utilizadas
- Python  
- PyTorch  
- CLIP (OpenAI)  
- FAISS  
- Pandas  
- PIL  


## Instalación de dependencias

En esta sección se instalan las librerías necesarias para:
- Manipulación de datos (`pandas`, `numpy`)
- Procesamiento de imágenes (`PIL`)
- Deep Learning (`torch`, `torchvision`)
- Búsqueda vectorial eficiente (`faiss`)


In [1]:
!pip -q install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install pandas numpy faiss-cpu



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install torch torchvision torchaudio



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


!pip -q install pillow


# Verificación de carga completa de `obras_arte` (MySQL → Pandas)

⚠️ **Advertencia importante**  
Este script **NO debe ejecutarse** si no se cuenta con:
- Acceso autorizado a la base de datos MySQL
- Credenciales válidas
- Permisos de lectura sobre la tabla `obras_arte`

Su ejecución sin acceso adecuado producirá errores de conexión o de permisos.

---

## Objetivo

Este documento describe un script de validación cuyo objetivo es **comprobar que la tabla `obras_arte` se lee completamente desde MySQL hacia un DataFrame de Pandas**, sin pérdida de registros.

Es especialmente útil cuando:
- La tabla es grande
- Se trabaja en VPS o conexiones remotas
- Se usan lecturas por chunks (`chunksize`)
- Se sospechan cortes silenciosos, timeouts o límites de memoria

---

## Estrategia general

El script sigue estos pasos:

1. **Confirmar la base de datos activa**  
   Evita errores comunes de conexión a una base equivocada.

2. **Obtener el conteo real de filas**  
   Usando `SELECT COUNT(*) FROM obras_arte`.

3. **Leer la tabla completa por chunks**  
   Para evitar cargar toda la tabla en memoria de una sola vez.

4. **Concatenar los chunks**  
   Construir un DataFrame final.

5. **Comparar resultados**  
   `len(df)` vs `COUNT(*)`.

---

## Requisitos previos

Antes de ejecutar este script, se debe verificar que:

- Se dispone de acceso válido a la base de datos MySQL
- Las credenciales en `DB_CONFIG` son correctas
- El usuario tiene permisos de lectura (`SELECT`)
- La tabla `obras_arte` existe en la base de datos activa

👉 **Si no se cumple alguno de estos puntos, el script no debe ejecutarse.**

---

## Configuración

### Parámetros de conexión

```python
DB_CONFIG = dict(
    host="",
    port=3306,
    user="",
    password="",
    database="",
    charset="",
    autocommit=False,
)


In [13]:
import pandas as pd
import mysql.connector

DB_CONFIG = dict(
    host="",
    port=3306,
    user="",
    password="",
    database="",
    charset="",
    autocommit=False,
)

CHUNKSIZE = 5_000

def main():
    conn = mysql.connector.connect(**DB_CONFIG)
    try:
        # 1) Confirmar a qué BD estás conectado
        db = pd.read_sql("SELECT DATABASE() AS db", conn).iloc[0]["db"]
        print("DB actual:", db)

        # 2) Conteo real de filas
        n_db = int(pd.read_sql("SELECT COUNT(*) AS n FROM obras_arte", conn).iloc[0]["n"])
        print("COUNT(*) en obras_arte:", n_db)

        # 3) Leer toda la tabla por chunks (consume TODO)
        query = "SELECT * FROM obras_arte"
        dfs = []
        total = 0

        for i, chunk in enumerate(pd.read_sql(query, conn, chunksize=CHUNKSIZE), start=1):
            dfs.append(chunk)
            total += len(chunk)
            if i == 1 or i % 5 == 0:
                print(f"Chunks leídos: {i} | filas acumuladas: {total}")

        df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

        # 4) Verificación final
        print("\n✅ DataFrame cargado")
        print("shape:", df.shape)
        print("len(df):", len(df))

        if len(df) != n_db:
            print("⚠️ OJO: len(df) != COUNT(*) (algo se cortó)")
        else:
            print("✅ OK: len(df) coincide con COUNT(*)")

        print(df.head(3))

    finally:
        conn.close()

if __name__ == "__main__":
    main()


ProgrammingError: 1045 (28000): Access denied for user 'ODBC'@'localhost' (using password: NO)

In [46]:
len(df)


23833

## Lectura de archivo de ejemplo (sin base de datos)

Cuando **no se tiene acceso a la base de datos** o se quiere **probar el notebook con un dataset local**, se puede usar un archivo CSV de ejemplo (por ejemplo `data_img.csv`) para validar el flujo de lectura en Pandas.

✅ **Ejecutar esta celda** si quieres trabajar **sin MySQL**.




In [5]:
import pandas as pd

df = pd.read_csv(
    "data_img.csv",
    sep=";",
    encoding="utf-8-sig",
    engine="python",      # más tolerante con CSVs imperfectos
    on_bad_lines="skip",  # o "warn" para ver avisos
)

print(df.shape)
print(df.head())

(23833, 28)
     id   source                                         source_url  \
0  4428  thyssen  https://www.museothyssen.org/coleccion/artista...   
1  4429  thyssen  https://www.museothyssen.org/coleccion/artista...   
2  4430  thyssen  https://www.museothyssen.org/coleccion/artista...   
3  4431  thyssen  https://www.museothyssen.org/coleccion/artista...   
4  4432  thyssen  https://www.museothyssen.org/coleccion/artista...   

  source_url_sha1                      inventory_no collection    room  \
0             ...              Nº INV.  ( DEC1656 )        NaN  Sala 2   
1             ...           Nº INV. 44.a ( 1931.4 )        NaN  Sala 2   
2             ...           Nº INV. 151 ( 1977.98 )        NaN  Sala 1   
3             ...  Nº INV. 44.d-e ( 1929.19.1-2.b )        NaN  Sala 2   
4             ...            Nº INV. 11 ( 1956.11 )        NaN  Sala 5   

                                            location  \
0  Thyssen-Bornemisza Collections, en depósito en...   
1   

In [6]:
df.columns

Index(['id', 'source', 'source_url', 'source_url_sha1', 'inventory_no',
       'collection', 'room', 'location', 'title', 'artist', 'date_text',
       'year_start', 'year_end', 'technique', 'medium', 'dimensions',
       'size_text', 'description', 'info_extra', 'bibliography',
       'exhibition_hist', 'image_url', 'image_full_url', 'raw_json',
       'scraped_at', 'updated_at', 'title_norm', 'artist_norm'],
      dtype='object')

## Carga del índice FAISS persistido

En esta sección se carga desde disco un índice FAISS previamente generado.
Este índice contiene los embeddings de las imágenes calculados con el modelo CLIP
y permite realizar búsquedas de similitud sin necesidad de recalcularlos.

El uso de un índice persistido:
- Reduce significativamente el tiempo de ejecución
- Permite reutilizar el modelo en distintos notebooks o aplicaciones
- Facilita la integración con sistemas externos (APIs, Streamlit, Web)

El archivo `.index` fue generado previamente y se carga utilizando `faiss.read_index`.

El índice FAISS almacena únicamente los vectores numéricos y sus posiciones,
por lo que la relación con las imágenes originales se mantiene mediante
una estructura externa de metadatos (por ejemplo, una lista o base de datos).


In [8]:
import os
import faiss

INDEX_PATH = r"C:\Users\David\PycharmProjects\proyectos_organizados\humanidades_digitales\analisis_de_imagen\clip_vitb32_combined.index"  # <-- especificar_ruta

INDEX_PATH = os.path.abspath(INDEX_PATH)
index = faiss.read_index(INDEX_PATH)
print("✅ Index cargado:", INDEX_PATH)
print("ntotal:", index.ntotal)


✅ Index cargado: C:\Users\David\PycharmProjects\proyectos_organizados\humanidades_digitales\analisis_de_imagen\clip_vitb32_combined.index
ntotal: 22634


## Recuperación de identificadores almacenados en FAISS

Los índices FAISS de tipo `IndexIDMap` o `IndexIDMap2` permiten asociar cada
vector almacenado con un identificador externo (por ejemplo, un ID de base
de datos, un índice de archivo o un identificador lógico).

En esta sección se recuperan los IDs persistidos dentro del índice FAISS.
Esto es fundamental para:
- Reconectar los resultados de la búsqueda con los archivos de imagen originales
- Mantener la trazabilidad entre vectores y metadatos externos
- Integrar FAISS con bases de datos o sistemas de archivo

Si el índice no fue creado como `IDMap`, esta operación no es posible y se
lanza una excepción.


Nota: En FAISS, el orden de los IDs recuperados corresponde al orden interno
de los vectores en el índice. Por esta razón, es indispensable mantener una
estructura externa de metadatos alineada con estos identificadores.



In [12]:
import numpy as np
import faiss

def get_faiss_ids(index) -> np.ndarray:
    # Para IndexIDMap / IndexIDMap2, FAISS guarda los ids en index.id_map
    if hasattr(index, "id_map"):
        return faiss.vector_to_array(index.id_map).astype(np.int64)
    raise TypeError("Este índice no tiene id_map (no es IDMap).")

ids_faiss = get_faiss_ids(index)
print("IDs en FAISS:", ids_faiss[:10], " ... total:", len(ids_faiss))


IDs en FAISS: [4455 4456 4457 4462 4463 4464 4465 4466 4467 4468]  ... total: 22634


## Alineación del índice FAISS con los metadatos del dataset

FAISS almacena únicamente vectores y sus identificadores, pero no conserva
información contextual como rutas de archivo, nombres o fuentes.
Por esta razón, es necesario reconstruir la relación entre los embeddings
y los metadatos originales.

En esta sección se:
1. Asegura que la columna `id` del DataFrame tenga un tipo numérico consistente
2. Reordena el DataFrame según el orden interno de los IDs almacenados en FAISS
3. Garantiza que cada vector del índice tenga su fila correspondiente en el dataset

El uso de `reindex(ids_faiss)` es fundamental para mantener la coherencia
entre los resultados de búsqueda y la información asociada a cada imagen.

Nota: La presencia de valores `NaN` indica que existen embeddings en el índice
FAISS que no tienen correspondencia directa en el archivo de metadatos.
Esto puede ocurrir si el índice fue generado a partir de múltiples fuentes
o si el dataset fue filtrado posteriormente.



In [13]:
some_id = dup.iloc[0]
print(df[df["id"] == some_id])


NameError: name 'dup' is not defined

In [14]:
import pandas as pd
import numpy as np

# 1) asegurar tipo
df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")

# 2) (opcional) quitar filas sin id
df = df[df["id"].notna()].copy()

# 3) detectar duplicados y resolverlos
dups = df["id"].duplicated(keep=False)
n_dups = int(dups.sum())

if n_dups > 0:
    print(f"⚠️ Hay {n_dups} filas con id duplicado (no se puede reindexar así).")
    print("Ejemplo de IDs duplicados:", df.loc[dups, "id"].unique()[:10])

    # CORRECCIÓN: nos quedamos con UNA fila por id
    # elige keep="last" o keep="first" según te convenga
    df = df.drop_duplicates(subset=["id"], keep="last").copy()
    print("✅ Duplicados resueltos. Filas después de dedupe:", len(df))

# 4) alinear según ids_faiss (respeta el orden)
ids_faiss_idx = pd.array(ids_faiss, dtype="Int64")

df_aligned = (
    df.set_index("id")
      .reindex(ids_faiss_idx)
      .reset_index()
)

print(df_aligned.head())
print("Filas alineadas:", len(df_aligned))
print("Faltantes en df:", df_aligned["source"].isna().sum())


     id   source                                         source_url  \
0  4455  thyssen  https://www.museothyssen.org/coleccion/artista...   
1  4456  thyssen  https://www.museothyssen.org/coleccion/artista...   
2  4457  thyssen  https://www.museothyssen.org/coleccion/artista...   
3  4462  thyssen  https://www.museothyssen.org/coleccion/artista...   
4  4463  thyssen  https://www.museothyssen.org/coleccion/artista...   

  source_url_sha1              inventory_no collection                room  \
0             ...   Nº INV. 210 ( 1929.13 )        NaN              Sala 2   
1             ...  Nº INV. 411 ( 1930.118 )        NaN              Sala 4   
2             ...    Nº INV. 162 ( 1966.6 )        NaN              Sala 1   
3             ...      Nº INV.  ( DEC1581 )        NaN  Sala no disponible   
4             ...    Nº INV. 261 ( 1931.2 )        NaN              Sala 3   

                                            location  \
0          Museo Nacional Thyssen-Bornemisza, Ma

## Filtrado del dataset a elementos presentes en el índice FAISS

En esta sección se construye un subconjunto del DataFrame original que
contiene únicamente las obras que están representadas dentro del índice FAISS.

Este filtrado permite:
- Evitar inconsistencias entre el dataset y el índice vectorial
- Asegurar que todas las filas del DataFrame tengan un embedding asociado
- Simplificar procesos posteriores de búsqueda y visualización

El criterio de inclusión se basa en la pertenencia del `id` al conjunto de
identificadores almacenados en FAISS.


In [15]:
df_in = df[df["id"].isin(ids_faiss)].copy()
print("Solo obras presentes en FAISS:", df_in.shape)


Solo obras presentes en FAISS: (22465, 28)


## Marcado de registros presentes en el índice FAISS

En esta sección se crea una columna indicadora (`in_faiss`) que señala si cada
registro del DataFrame tiene un embedding almacenado en el índice FAISS.

Este enfoque permite:
- Mantener el dataset completo sin descartar información
- Diferenciar claramente qué obras participan en la búsqueda vectorial
- Facilitar análisis comparativos, filtros dinámicos o visualizaciones

La columna `in_faiss` toma el valor:
- `1` si el `id` del registro está presente en FAISS
- `0` en caso contrario


In [17]:
s = set(ids_faiss.tolist())
df["in_faiss"] = df["id"].astype("Int64").apply(lambda x: int(x in s) if pd.notna(x) else 0)


## Búsqueda texto → imagen con CLIP + FAISS

En esta sección se habilita la búsqueda de obras a partir de una consulta en texto
(por ejemplo: “paisaje nocturno”, “retrato cubista”, “caballo”, etc.).

El flujo es:

1. **Carga del modelo CLIP** (`openai/clip-vit-base-patch32`) y selección automática de dispositivo
   (`cuda` si hay GPU disponible, si no `cpu`).
2. **Generación del embedding del texto**:
   - Se tokeniza el texto con `CLIPProcessor`
   - Se obtiene la representación del encoder de texto (`model.text_model`)
   - Se proyecta al espacio común de CLIP (512 dimensiones) con `text_projection`
   - Se **normaliza** el vector para que la similitud por producto punto (IP) sea equivalente a coseno
3. **Consulta en FAISS**:
   - `index.search(q, k)` devuelve los `k` resultados más similares
   - `I` contiene los IDs (o posiciones) recuperados
   - `D` contiene los scores de similitud
4. **Reconstrucción con metadatos**:
   - Se mapean los IDs recuperados al DataFrame (`df`) para devolver resultados interpretables
   - Se filtran IDs que no existan en el DataFrame, si los hubiera


In [18]:
import numpy as np
import pandas as pd
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

def text_embedding(text: str) -> np.ndarray:
    tok = processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    tok = {k: v.to(device) for k, v in tok.items()}

    with torch.inference_mode():
        # 1) pasa SOLO por el encoder de texto
        text_out = model.text_model(
            input_ids=tok["input_ids"],
            attention_mask=tok.get("attention_mask", None),
        )
        # 2) pooler_output -> proyección CLIP a 512
        v = text_out.pooler_output
        v = model.text_projection(v)

        # 3) normalizar para IndexFlatIP
        v = v / v.norm(dim=-1, keepdim=True)

    return v.detach().cpu().numpy().astype("float32")  # (1,512)

def search_df(query: str, k: int = 10) -> pd.DataFrame:
    q = text_embedding(query)             # (1,512)
    D, I = index.search(q, k)             # I: (1,k) ids de obras, D: scores
    ids = I[0].astype("int64")
    scores = D[0]

    # OJO: si algún id no está en df, lo filtramos
    df_idx = df.set_index("id")
    keep = [int(x) for x in ids if int(x) in df_idx.index]

    out = (df_idx.loc[keep]
                 .assign(score=scores[:len(keep)])
                 .reset_index()
                 .sort_values("score", ascending=False))
    return out


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


## Ejecución de la consulta y visualización de resultados

En esta sección se permite al usuario introducir una consulta textual que será
utilizada para buscar obras visualmente relacionadas dentro del índice FAISS.

El flujo es el siguiente:
- El usuario ingresa un texto libre (consulta semántica)
- Se ejecuta la función de búsqueda `search_df`
- Se recuperan las `k` obras más similares
- Se muestran los principales campos descriptivos junto con el score de similitud

El valor `score` representa el grado de similitud entre el embedding del texto
y el embedding de la imagen; valores más altos indican mayor afinidad semántica.

Este mecanismo demuestra cómo es posible consultar un archivo visual a partir
de lenguaje natural, habilitando nuevas formas de exploración, curaduría y
análisis semántico en colecciones de imágenes.



In [19]:
qtext = input("🔎 Buscar: ")   
df_results = search_df(qtext, k=15)

df_results[["id","title","artist","score","image_url"]].head(15)

🔎 Buscar:  dancing


,id,title,artist,score,image_url
14,10563,"La Java, 1925",Georges Emile Capon,1.468019,https://www.artic.edu/iiif/2/e127aa5e-2154-511...
13,12700,"Capitan Babeo - Cucuba, plate 16 from Balli di...",Jacques Callot,1.466939,https://www.artic.edu/iiif/2/daa7e629-9b5f-334...
12,8057,"Three Studies of a Dancer in Fourth Position, ...",Hilaire Germain Edgar Degas,1.466630,https://www.artic.edu/iiif/2/041d1ff9-e64a-eee...
11,13024,The Actor Sakata Hangoro III as the Guard Yaha...,Katsukawa Shun’en,1.465556,https://www.artic.edu/iiif/2/82ee5f78-facf-d8b...
10,14639,"New Years Eve Party I, n.d.",Francis Chapin,1.463691,https://www.artic.edu/iiif/2/f8376219-b48a-2b4...
9,7174,"Alice, 1892",William Merritt Chase,1.463109,https://www.artic.edu/iiif/2/91fcc71a-50eb-f41...
8,6701,"Croquet Scene, 1866",Winslow Homer,1.460814,https://www.artic.edu/iiif/2/f7b2db45-f393-653...
7,8581,"Be Careful with that Step!, 1816/1820",Francisco José de Goya y Lucientes,1.457967,https://www.artic.edu/iiif/2/626bcfe3-6a1b-281...
6,7378,"The Star, 1879/81",Hilaire Germain Edgar Degas,1.457958,https://www.artic.edu/iiif/2/0850f3ab-a29d-acc...
5,10436,"Dancer Bending Forward, 1874/79",Hilaire Germain Edgar Degas,1.454156,https://www.artic.edu/iiif/2/fdb26db9-c4cd-96f...


## Visualización de los resultados recuperados

En esta sección se presentan visualmente las obras más relevantes devueltas por
la búsqueda semántica texto → imagen.

Para cada resultado se muestra:
- El título de la obra
- El artista
- El score de similitud semántica
- La imagen asociada (si está disponible)

Esta visualización permite evaluar de forma cualitativa la coherencia entre
la consulta textual y los resultados obtenidos, complementando el análisis
numérico con una inspección visual directa.


In [20]:
from IPython.display import Image, display

top = df_results.head(10)

for i, row in top.iterrows():
    print(f"{row['title']} — {row['artist']} (score={row['score']:.3f})")
    if pd.notna(row["image_url"]):
        display(Image(url=row["image_url"], width=300))
    print("-" * 60)


La Java, 1925 — Georges Emile Capon (score=1.468)


------------------------------------------------------------
Capitan Babeo - Cucuba, plate 16 from Balli di Sfessania, c. 1622 — Jacques Callot (score=1.467)


------------------------------------------------------------
Three Studies of a Dancer in Fourth Position, 1879/80 — Hilaire Germain Edgar Degas (score=1.467)


------------------------------------------------------------
The Actor Sakata Hangoro III as the Guard Yahazu no Yadahei in the Play Otokoyama O-Edo no Ishizue, Performed at the Kiri Theater in the Eleventh Month, 1794, c. 1794 — Katsukawa Shun’en (score=1.466)


------------------------------------------------------------
New Years Eve Party I, n.d. — Francis Chapin (score=1.464)


...

------------------------------------------------------------
Croquet Scene, 1866 — Winslow Homer (score=1.461)


------------------------------------------------------------
Be Careful with that Step!, 1816/1820 — Francisco José de Goya y Lucientes (score=1.458)


------------------------------------------------------------
The Star, 1879/81 — Hilaire Germain Edgar Degas (score=1.458)


------------------------------------------------------------
Dancer Bending Forward, 1874/79 — Hilaire Germain Edgar Degas (score=1.454)


------------------------------------------------------------


## Búsqueda imagen → imagen con CLIP + FAISS

En esta sección se implementa la recuperación de imágenes similares a partir de
una imagen de consulta (búsqueda visual).

### Idea principal
1. Se toma una **imagen local** como entrada.
2. Se genera su **embedding CLIP** (vector de 512 dimensiones) usando el encoder visual.
3. Se **normaliza** el embedding para que el producto punto (Inner Product) se comporte
   como similitud coseno (esto es lo habitual cuando el índice FAISS es `IndexFlatIP`).
4. Se consulta el índice FAISS (`index.search`) para recuperar las `k` imágenes más similares.
5. Se reconstruyen los resultados usando los **metadatos** almacenados en el DataFrame (`df`).

### Funciones incluidas
- `image_embedding_from_path(img_path)`: genera el embedding de una imagen local y lo normaliza.
- `search_by_image(img_path, k)`: realiza la búsqueda en FAISS y retorna un DataFrame con:
  `id`, `title`, `artist`, `image_url` y `score`.

### Nota sobre IDs y metadatos
FAISS no almacena metadatos (título, artista, URL).  
Por eso, los resultados deben mapearse a `df` mediante el campo `id`.
El mapeo asume un caso típico donde el índice es un `IDMap` y devuelve IDs reales
que coinciden con `df["id"]`. Si el índice devuelve posiciones internas, se requiere
una tabla de correspondencia adicional.


In [31]:
# =========================
# IMAGEN -> IMAGEN (CLIP + FAISS)
# =========================

import numpy as np
import pandas as pd
import torch
from PIL import Image

# Requisitos previos (ya los tienes en tu notebook):
# - df: DataFrame con columna 'id' (int) y campos como title/artist/image_url
# - index: índice FAISS ya cargado
# - model, processor: CLIPModel/CLIPProcessor cargados
# - device: "cuda" o "cpu"

def image_embedding_from_path(img_path: str) -> np.ndarray:
    """Genera embedding CLIP (1,512) para una imagen local y lo normaliza."""
    img = Image.open(img_path).convert("RGB")

    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # capa de proyección (según versión/modelo)
    proj = getattr(model, "visual_projection", None) or getattr(model, "vision_projection", None)
    if proj is None:
        raise AttributeError("No encuentro visual_projection/vision_projection en el modelo CLIP.")

    with torch.inference_mode():
        # encoder de visión (salida completa)
        vision_out = model.vision_model(**inputs)

        # pooler_output -> proyección a 512
        v = proj(vision_out.pooler_output)  # (1,512)

        # normalizar para IndexFlatIP (cosine)
        v = v / v.norm(dim=-1, keepdim=True)

    return v.detach().cpu().numpy().astype("float32")  # (1,512)

def search_by_image(img_path: str, k: int = 10) -> pd.DataFrame:
    """
    Busca las k más similares en el índice FAISS usando una imagen externa.
    Devuelve un DataFrame con score y metadatos del df.
    """
    q = image_embedding_from_path(img_path)   # (1,512)
    D, I = index.search(q, k)                 # I: (1,k) ids o posiciones; D: (1,k) scores

    ids = I[0].astype("int64")
    scores = D[0]

    # Caso típico: tu índice es IDMap y devuelve ids reales = df['id']
    df_idx = df.set_index("id")
    keep_ids = [int(x) for x in ids if int(x) in df_idx.index]

    out = (df_idx.loc[keep_ids]
           .assign(score=scores[:len(keep_ids)])
           .reset_index()
           .sort_values("score", ascending=False))
    return out

# ---- USO ----
img_path = r"C:\Users\David\PycharmProjects\proyectos_organizados\humanidades_digitales\analisis_de_imagen\djkbflhikuadslsad.jpg"
df_sim = search_by_image(img_path, k=200)
display(df_sim[["id","title","artist","score","image_url"]].head(200))

# ---- MOSTRAR TOP IMÁGENES (opcional) ----
from IPython.display import Image as IPyImage, display

top = df_sim.head(60)
for _, row in top.iterrows():
    print(f"ID: {row.get('id','')} - Title: {row.get('title','')} — {row.get('artist','')} (score={row['score']:.3f})")
    if pd.notna(row.get("image_url", None)):
        display(IPyImage(url=row["image_url"], width=300))
    print("-" * 60)





,id,title,artist,score,image_url
198,33194,"Worker Shabti of Henettawy (C), Daughter of Is...",NaN,0.454882,https://images.metmuseum.org/CRDImages/eg/web-...
197,33269,"Worker Shabti of Henettawy (C), Daughter of Is...",NaN,0.454742,https://images.metmuseum.org/CRDImages/eg/web-...
196,20898,Winged scarab,NaN,0.454215,https://images.metmuseum.org/CRDImages/eg/web-...
195,19892,Scarab Inscribed with the Throne Name of Thutm...,NaN,0.453973,https://images.metmuseum.org/CRDImages/eg/web-...
194,19031,Osiris-Iah,NaN,0.453860,https://images.metmuseum.org/CRDImages/eg/web-...
...,...,...,...,...,...
4,19488,Scarab Finger Ring,NaN,0.369462,https://images.metmuseum.org/CRDImages/eg/web-...
3,29710,Scarab Inscribed with a Hieroglyphic Motif,NaN,0.360331,https://images.metmuseum.org/CRDImages/eg/web-...
2,19739,Shell Pendant with the Name of Senwosret II,NaN,0.350826,https://images.metmuseum.org/CRDImages/eg/web-...
1,23273,Gold Base Plate of a Scarab of an Official,NaN,0.315762,https://images.metmuseum.org/CRDImages/eg/web-...


ID: 33194 - Title: Worker Shabti of Henettawy (C), Daughter of Isetemkheb — nan (score=0.455)


------------------------------------------------------------
ID: 33269 - Title: Worker Shabti of Henettawy (C), Daughter of Isetemkheb — nan (score=0.455)


------------------------------------------------------------
ID: 20898 - Title: Winged scarab — nan (score=0.454)


------------------------------------------------------------
ID: 19892 - Title: Scarab Inscribed with the Throne Name of Thutmose III — nan (score=0.454)


...

------------------------------------------------------------
ID: 16938 - Title: Scarab Incised with Hieroglyphs — nan (score=0.448)


------------------------------------------------------------
ID: 33311 - Title: Worker Shabti of Henettawy (C), Daughter of Isetemkheb — nan (score=0.448)


------------------------------------------------------------
ID: 32733 - Title: Pakhet (?) Amulet — nan (score=0.448)


------------------------------------------------------------
ID: 29693 - Title: Scarab Inscribed with a Geometric Pattern — nan (score=0.447)


------------------------------------------------------------


# Búsqueda semántica CLIP + FAISS con recuperación de metadata desde MySQL

**nota:** solo sirve con acceso a base de datos

Este bloque implementa un flujo completo de **búsqueda semántica por texto** sobre un índice FAISS (CLIP), enlazando los resultados con **metadata estructurada almacenada en MySQL** y mostrando resultados visuales.

El objetivo es:
- Buscar similitud semántica en el índice vectorial
- Recuperar los `obra_id` reales
- Consultar la base de datos relacional
- Presentar resultados interpretables (título, artista, imagen, score)

---

## 1. Verificación del tipo de índice FAISS (IDMap)

```python
print("Tiene id_map:", hasattr(index, "id_map"))


In [21]:
import os
import pandas as pd
import mysql.connector
from IPython.display import Image as IPyImage, display

# -------------------------
# 1) CHECK: ¿tu índice tiene IDs reales?
# -------------------------
print("Tiene id_map:", hasattr(index, "id_map"))
if not hasattr(index, "id_map"):
    print("⚠️ OJO: tu índice NO parece IDMap. Los números de FAISS serían POSICIONES, no obra_id.")

# -------------------------
# 2) MySQL connection (usa env vars)
# -------------------------
MYSQL_HOST = os.getenv("MYSQL_HOST", "")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", ""))
MYSQL_USER = os.getenv("MYSQL_USER", "")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD","") 
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "")

def mysql_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE,
        autocommit=True
    )

# -------------------------
# 3) Traer obras por ids (batch IN)
# -------------------------
def fetch_obras_by_ids(ids):
    ids = [int(x) for x in ids if int(x) != -1]
    if not ids:
        return pd.DataFrame()

    placeholders = ",".join(["%s"] * len(ids))
    sql = f"""
    SELECT
        id,
        source,
        title,
        artist,
        date_text,
        image_full_url,
        image_url,
        source_url
    FROM obras_arte
    WHERE id IN ({placeholders})
    """

    conn = mysql_conn()
    try:
        cur = conn.cursor(dictionary=True)
        cur.execute(sql, ids)
        rows = cur.fetchall() or []
        return pd.DataFrame(rows)
    finally:
        try:
            cur.close()
        except Exception:
            pass
        conn.close()

# -------------------------
# 4) Buscar texto -> ids -> traer metadata -> mostrar
# -------------------------
def search_text_with_mysql(query: str, k: int = 25, show_images: int = 10) -> pd.DataFrame:
    # results = [(id, score), ...]
    results = search_text(query, k=k)

    ids = [rid for rid, _ in results]
    score_map = {int(rid): float(score) for rid, score in results}

    df_meta = fetch_obras_by_ids(ids)
    if df_meta.empty:
        print("No encontré filas en MySQL para esos IDs.")
        return df_meta

    # Añadir score y ordenar por score
    df_meta["score"] = df_meta["id"].map(score_map)
    df_meta = df_meta.sort_values("score", ascending=False)

    display(df_meta[["id","title","artist","score","image_full_url","image_url","source"]].head(k))

    # Mostrar imágenes (image_full_url fallback a image_url)
    n = min(show_images, len(df_meta))
    for _, row in df_meta.head(n).iterrows():
        url = row.get("image_full_url") or row.get("image_url")
        print(f'{row.get("title","")} — {row.get("artist","")}  (score={row["score"]:.4f})')
        if url and isinstance(url, str) and url.strip():
            display(IPyImage(url=url, width=320))
        else:
            print("   [sin imagen]")
        print("-" * 60)

    return df_meta

# ---- USO ----
df_res = search_text_with_mysql("personas bailando", k=25, show_images=10)


Tiene id_map: True


ValueError: invalid literal for int() with base 10: ''